# Trajectory Analysis for Weight Scan

In [1]:
0

0

In [2]:
import sys
working_directory = "/home/icb/kemal.inecik/work/codes/sctram"
sys.path.append(working_directory)

import logging
import subprocess
import gc
import os
import time
import numpy as np
import pandas as pd
import networkx as nx
import scanpy as sc
import anndata as ad
import pickle
import itertools
import torch
import tqdm
import scvi
import h5py
from scipy import stats

sc.settings.verbose = 3

In [3]:
from sctram.api._lower_level import TrajectoryEvaluationAPI
from sctram.generate.real import sc_suo_developmental_complete
from sctram.input import InputTrajectories

incremental_analysis_path = "/home/icb/kemal.inecik/work/codes/sctram/reproducibility/server_sync/experiments"
sys.path.append(incremental_analysis_path)
from incremental_training.helper.constants import metrics_scib, metric_direction_dict

2025-05-09 20:13:55.426 | INFO     | sctram.api._defaults_read:load_default_metrics:23 - Loaded default metrics from /home/icb/kemal.inecik/work/codes/sctram/sctram/api/_defaults.yaml
2025-05-09 20:13:55.427 | INFO     | sctram.api._defaults_read:load_default_metrics:79 - Default metrics YAML structure validated successfully.


In [4]:
# Important to have consistent figures across platforms

%matplotlib inline
%config InlineBackend.figure_format='retina'

import pickle

from networkx.drawing.nx_agraph import graphviz_layout
from matplotlib import gridspec
import matplotlib.pyplot as plt
import seaborn as sns
import colorcet as cc
from adjustText import adjust_text  
import matplotlib.patheffects as path_effects

_rcparams_path = os.path.join(working_directory, "reproducibility/figure_rcparams/rcparams.pickle")
with open(_rcparams_path, "rb") as file:
    _rcparams = pickle.load(file)
plt.rcParams.update(_rcparams)

In [5]:
print(f"CUDA used: {torch.cuda.is_available()}")

dataset_dir = "/home/icb/kemal.inecik/lustre_workspace/temp_sctram_data"
helpers_directory = os.path.join(os.getcwd(), "helper")
logs_directory = os.path.join(os.getcwd(), "logs")

CUDA used: True


In [6]:
definitions_path = os.path.join(dataset_dir, f"weight_scan_analysis_definitions_dict.pkl")
with open(definitions_path, "rb") as _file_definitions:
    definitions = pickle.load(_file_definitions)    

In [7]:
definitions

{'tardis_sciplex_whole': {'training_anndata_path': '/home/icb/kemal.inecik/lustre_workspace/temp_sctram_data/tardis_cpa_sciplex_v2_encode_incremental_training_anndata.h5ad',
  'vae_path': 'placeholder',
  'pickle_paths_path': '/home/icb/kemal.inecik/lustre_workspace/temp_sctram_data/tardis_cpa_sciplex_v2_encode_weight_scan_latent_paths_whole.pkl',
  'trajectory_object_path': '/home/icb/kemal.inecik/lustre_workspace/temp_sctram_data/tardis_cpa_sciplex_v2_encode_weight_scan_trajectory_object_whole.pkl',
  'label_obs_column': 'drug_dose_name',
  'batch_obs_column': 'none'},
 'tardis_sciplex_dose': {'training_anndata_path': '/home/icb/kemal.inecik/lustre_workspace/temp_sctram_data/tardis_cpa_sciplex_v2_encode_incremental_training_anndata.h5ad',
  'vae_path': 'placeholder',
  'pickle_paths_path': '/home/icb/kemal.inecik/lustre_workspace/temp_sctram_data/tardis_cpa_sciplex_v2_encode_weight_scan_latent_paths_dose.pkl',
  'trajectory_object_path': '/home/icb/kemal.inecik/lustre_workspace/temp_

# Loading the benchmarking

In [8]:
sort_columns_by = ["run_id", "trajectory", "weight_coef", "path", "metric", "score"]

In [9]:
df_sctram = pd.DataFrame()
count = 0

for run_id in definitions.keys():

    pickle_paths_path = definitions[run_id]["pickle_paths_path"]
    with open(pickle_paths_path, "rb") as _file_pickle_paths:
        steps_paths_dict = pickle.load(_file_pickle_paths)    
    trajectory_path = definitions[run_id]["trajectory_object_path"]
    
    for weight_coef, latent_object_path in steps_paths_dict.items():
        
        with open(trajectory_path, "rb") as _file_trajectory_path:
            trajectories_object = pickle.load(_file_trajectory_path)
        
        for trajectory in sorted(trajectories_object.graph["trajectories"]):
            
            output_file_path = os.path.join(dataset_dir, f"metric_sctram_{run_id}_weight_{weight_coef}_trajectory_{trajectory}.pkl")
            if os.path.exists(output_file_path) and os.path.isfile(output_file_path):
            
                df_trajectory_obsm = pd.read_pickle(output_file_path)
                
                df_trajectory_obsm["run_id"] = run_id
                df_trajectory_obsm["weight_coef"] = weight_coef
                df_trajectory_obsm["trajectory"] = trajectory
                df_sctram = pd.concat([df_sctram, df_trajectory_obsm])
                count += 1
            else:
                print(f"Not calculated: metric_sctram_{run_id}_weight_{weight_coef}_trajectory_{trajectory}")

print(f" - Number of jobs obtained: {count}")

df_sctram.reset_index(drop=True, inplace=True)
df_sctram['score'] = pd.to_numeric(df_sctram['score'], errors='raise')
assert set(df_sctram["metric"].unique()) == set(metric_direction_dict.keys())

df_sctram = df_sctram[sort_columns_by]
df_sctram.sort_values(by=sort_columns_by, inplace=True, ignore_index=True)
df_sctram.reset_index(drop=True, inplace=True)

df_sctram

Not calculated: metric_sctram_tardis_sciplex_whole_weight_0.95_trajectory_fork
Not calculated: metric_sctram_tardis_sciplex_whole_weight_0.95_trajectory_linear_alternative
Not calculated: metric_sctram_tardis_sciplex_whole_weight_0.925_trajectory_fork
Not calculated: metric_sctram_tardis_sciplex_whole_weight_0.85_trajectory_fork
Not calculated: metric_sctram_tardis_sciplex_whole_weight_0.825_trajectory_fork
Not calculated: metric_sctram_tardis_sciplex_whole_weight_0.8_trajectory_fork
Not calculated: metric_sctram_tardis_sciplex_whole_weight_0.8_trajectory_linear_alternative
Not calculated: metric_sctram_tardis_sciplex_whole_weight_0.75_trajectory_fork
Not calculated: metric_sctram_tardis_sciplex_whole_weight_0.75_trajectory_linear_alternative
Not calculated: metric_sctram_tardis_sciplex_whole_weight_0.725_trajectory_fork
Not calculated: metric_sctram_tardis_sciplex_whole_weight_0.7_trajectory_linear
Not calculated: metric_sctram_tardis_sciplex_whole_weight_0.675_trajectory_fork
Not cal

,run_id,trajectory,weight_coef,path,metric,score
0,tardis_sciplex_dose,linear,0.025,adjacency,accuracy,0.343750
1,tardis_sciplex_dose,linear,0.025,adjacency,average_shortest_path_difference,2.358880
2,tardis_sciplex_dose,linear,0.025,adjacency,clustering_coeff_diff,0.660470
3,tardis_sciplex_dose,linear,0.025,adjacency,f1_score,0.400000
4,tardis_sciplex_dose,linear,0.025,adjacency,frobenius,4.074300
...,...,...,...,...,...,...
24475,tardis_sciplex_whole,linear_alternative,1.000,pseudotime,pearson_correlation,NaN
24476,tardis_sciplex_whole,linear_alternative,1.000,pseudotime,r_squared,NaN
24477,tardis_sciplex_whole,linear_alternative,1.000,pseudotime,r_squared_with_spline,NaN
24478,tardis_sciplex_whole,linear_alternative,1.000,pseudotime,spearman_correlation,-0.109846


# Analysis

In [10]:
0

0